In [1]:
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import List, Optional
import os
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [4]:
JUDGE_SYSTEM_PROMPT = """\
You are a strict evaluator of energy-market answers.
Follow expert industry standards.
Your output MUST exactly match the provided output schema.
Do not add extra fields or surrounding text.
"""

In [5]:
class AccuracyResult(BaseModel):
    accuracy_score: float = Field(ge=0.0, le=1.0)
    reasoning: str


class ApproachResult(BaseModel):
    approach_correctness: int = Field(ge=1, le=5)
    reasoning: str


class SourceResult(BaseModel):
    source_validity: int = Field(ge=1, le=5)
    reasoning: str


class AttributeDetail(BaseModel):
    name: str
    expected: str
    found: bool
    agent_value: Optional[str] = None


class AttributeAlignmentResult(BaseModel):
    total_attributes: int
    matched_attributes: int
    alignment_score: float = Field(ge=0.0, le=1.0)
    attribute_details: List[AttributeDetail]
    reasoning: str

In [6]:
APPROACH_PROMPT = """\
You are evaluating the approach correctness of an AI agent's answer to an energy-market question.

Question:
{question}

Expected Answer (Ground Truth):
{expected_answer}

Agent's Answer:
{agent_answer}

Evaluate:
- Correct problem framing
- Appropriate data sources (ISO postings, tariffs, settlement data, APIs)
- Logical analytical steps
- Correct tool usage (if applicable)

Rating scale:
5=expert-like, 4=minor issues, 3=notable gaps, 2=major flaws, 1=wrong approach
"""

In [7]:
ACCURACY_PROMPT = """\
You are evaluating the factual and numerical accuracy of an AI agent's answer to an energy-market question.

Question:
{question}

Expected Answer (Ground Truth):
{expected_answer}

Agent's Answer:
{agent_answer}

Evaluate:
- Numerical correctness (values, sign, magnitude, units, time basis)
- Factual alignment (market/ISO, node/zone, product, settlement type)
- Completeness of key facts

Tolerance:
- Allow <= {abs_tol} absolute error OR <= {rel_tol}% relative error unless exactness is required.
"""

In [8]:
SOURCE_PROMPT = """\
You are evaluating the source validity of an AI agent's answer to an energy-market question.

Question:
{question}

Valid / Expected Sources:
{valid_sources}

Agent's Answer:
{agent_answer}

Evaluate:
- Authority of sources
- Alignment with expected sources
- Appropriateness for the claim
- Missing citations when required
"""

In [9]:
ATTRIBUTE_PROMPT = """\
You are evaluating attribute alignment of an AI agent's answer against expected attributes.

Question:
{question}

Expected Attributes:
{attributes_section}

Agent's Answer:
{agent_answer}

For each expected attribute, decide whether the agent answer contains the correct value
or a reasonable equivalent, respecting units and time basis.
"""

In [10]:
def judge_accuracy(
    question: str,
    expected_answer: str,
    agent_answer: str,
    *,
    abs_tol: float = 0.01,
    rel_tol: float = 0.5,
    model: str = "gpt-4o-mini",
) -> AccuracyResult:
    prompt = ACCURACY_PROMPT.format(
        question=question,
        expected_answer=expected_answer,
        agent_answer=agent_answer,
        abs_tol=abs_tol,
        rel_tol=rel_tol,
    )
    resp = client.responses.parse(
        model=model,
        input=[
            {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ],
        text_format=AccuracyResult,
    )
    return resp.output_parsed

In [11]:
def judge_approach(
    question: str,
    expected_answer: str,
    agent_answer: str,
    *,
    model: str = "gpt-4o-mini",
) -> ApproachResult:
    prompt = APPROACH_PROMPT.format(
        question=question,
        expected_answer=expected_answer,
        agent_answer=agent_answer,
    )
    resp = client.responses.parse(
        model=model,
        input=[
            {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ],
        text_format=ApproachResult,
    )
    return resp.output_parsed

In [15]:
def judge_sources(
    question: str,
    valid_sources: str,
    agent_answer: str,
    *,
    model: str = "gpt-4o-mini",
) -> SourceResult:
    prompt = SOURCE_PROMPT.format(
        question=question,
        valid_sources=valid_sources,
        agent_answer=agent_answer,
    )
    resp = client.responses.parse(
        model=model,
        input=[
            {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ],
        text_format=SourceResult,
    )
    return resp.output_parsed

In [16]:
def judge_attributes(
    question: str,
    attributes_section: str,
    agent_answer: str,
    *,
    model: str = "gpt-4o-mini",
) -> AttributeAlignmentResult:
    prompt = ATTRIBUTE_PROMPT.format(
        question=question,
        attributes_section=attributes_section,
        agent_answer=agent_answer,
    )
    resp = client.responses.parse(
        model=model,
        input=[
            {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ],
        text_format=AttributeAlignmentResult,
    )
    return resp.output_parsed

In [17]:
question = "What was the DA LMP at HB_HOUSTON for HE 14 on 2025-08-01?"
ground_truth = "DA LMP at HB_HOUSTON for HE 14 on 2025-08-01 was $52.37/MWh."
answer = "It was about $52.4/MWh at HB_HOUSTON for hour ending 14 on 2025-08-01."

print(judge_accuracy(question, ground_truth, answer))

accuracy_score=1.0 reasoning="The agent's answer of $52.4/MWh is within the tolerance limit of the expected answer of $52.37/MWh, both in terms of numerical correctness and factual alignment. The statement correctly references HB_HOUSTON and hour ending 14 on the specified date, maintaining completeness of key facts."


In [18]:
print(judge_approach(question, ground_truth, answer))

approach_correctness=4 reasoning='The agent provided a reasonable approximation of the DA LMP with only a minor rounding difference. The framing of the problem was appropriate, and while no specific data source was cited, the general information aligns with industry standards. The answer is logically sound but lacks direct reference to data sources.'


In [19]:
print(judge_sources(question, "ERCOT Day-Ahead Settlement Prices / ERCOT API", answer))

source_validity=1 reasoning="The agent's answer provides a specific price point for the DA LMP at HB_HOUSTON, which aligns with data typically found in the ERCOT Day-Ahead Settlement Prices or the ERCOT API. However, it lacks direct citation of these expected sources, which diminishes the credibility of the claim. Overall, while the information appears accurate, the missing citation affects the authority."
